# ML-Enhanced IDS — Training & Evaluation Notebook

This notebook walks through the complete ML pipeline:
1. Data loading & exploration
2. Model training (RF, XGBoost, DNN)
3. Ensemble stacking
4. Evaluation & visualization

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Suppress TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Add project root to path
project_root = str(Path.cwd().parent) if 'notebooks' in str(Path.cwd()) else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.feature_config import *
from src.models.model_utils import *

plt.style.use('seaborn-v0_8-darkgrid')
print('Setup complete!')

## 1. Load Processed Data

In [ ]:
data = load_processed_data(os.path.join(project_root, 'data/processed'))
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Classes: {np.unique(y_train)}')
print(f'Features: {X_train.shape[1]}')

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y, title in [(axes[0], y_train, 'Train'), (axes[1], y_test, 'Test')]:
    unique, counts = np.unique(y, return_counts=True)
    labels = [ID_TO_CLASS.get(u, f'Class_{u}') for u in unique]
    colors = plt.cm.Set2(np.linspace(0, 1, len(unique)))
    ax.bar(labels, counts, color=colors, edgecolor='white')
    ax.set_title(f'{title} Set Distribution')
    ax.set_ylabel('Count')
    for i, (l, c) in enumerate(zip(labels, counts)):
        ax.text(i, c + max(counts)*0.01, f'{c:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 2. Train Individual Models

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
import time

start = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5,
                            min_samples_leaf=2, max_features='sqrt',
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_time = time.time() - start

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)
rf_metrics = evaluate_model(y_test, rf_pred, rf_prob, 'Random Forest')
print(f'Training time: {rf_time:.1f}s')

In [ ]:
# XGBoost
from xgboost import XGBClassifier
from collections import Counter

counter = Counter(y_train)
total = len(y_train)
n_cls = len(counter)
sample_weights = np.array([total / (n_cls * counter[yi]) for yi in y_train])

start = time.time()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                     objective='multi:softprob', num_class=NUM_CLASSES,
                     random_state=42, eval_metric='mlogloss',
                     use_label_encoder=False, n_jobs=-1)
xgb.fit(X_train, y_train, sample_weight=sample_weights)
xgb_time = time.time() - start

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)
xgb_metrics = evaluate_model(y_test, xgb_pred, xgb_prob, 'XGBoost')
print(f'Training time: {xgb_time:.1f}s')

In [ ]:
# DNN
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.utils.class_weight import compute_class_weight

input_dim = X_train.shape[1]
dnn = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.BatchNormalization(), layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

dnn.compile(optimizer=keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

unique_cls = np.unique(y_train)
cw = compute_class_weight('balanced', classes=unique_cls, y=y_train)
cw_dict = dict(zip(unique_cls, cw))

start = time.time()
history = dnn.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=256,
                  class_weight=cw_dict, callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)], verbose=1)
dnn_time = time.time() - start

dnn_prob = dnn.predict(X_test, verbose=0)
dnn_pred = np.argmax(dnn_prob, axis=1)
dnn_metrics = evaluate_model(y_test, dnn_pred, dnn_prob, 'DNN')
print(f'Training time: {dnn_time:.1f}s')

## 3. Stacking Ensemble

In [ ]:
from sklearn.linear_model import LogisticRegression

def pad(probs, n):
    if probs.shape[1] < n:
        p = np.zeros((probs.shape[0], n))
        p[:, :probs.shape[1]] = probs
        return p
    return probs

# Generate meta-features
rf_prob_train = pad(rf.predict_proba(X_train), NUM_CLASSES)
xgb_prob_train = pad(xgb.predict_proba(X_train), NUM_CLASSES)
dnn_prob_train = pad(dnn.predict(X_train, verbose=0), NUM_CLASSES)
meta_X_train = np.hstack([rf_prob_train, xgb_prob_train, dnn_prob_train])

rf_prob_test = pad(rf_prob, NUM_CLASSES)
xgb_prob_test = pad(xgb_prob, NUM_CLASSES)
dnn_prob_test = pad(dnn_prob, NUM_CLASSES)
meta_X_test = np.hstack([rf_prob_test, xgb_prob_test, dnn_prob_test])

# Train meta-classifier
meta = LogisticRegression(max_iter=500, multi_class='multinomial', solver='lbfgs',
                           class_weight='balanced', random_state=42)
meta.fit(meta_X_train, y_train)

ens_pred = meta.predict(meta_X_test)
ens_prob = meta.predict_proba(meta_X_test)
ens_metrics = evaluate_model(y_test, ens_pred, ens_prob, 'Stacking Ensemble')

## 4. Model Comparison

In [ ]:
# Comparison table
models = {'Random Forest': rf_metrics, 'XGBoost': xgb_metrics, 'DNN': dnn_metrics, 'Ensemble': ens_metrics}
comparison = pd.DataFrame({
    name: {k: v for k, v in m.items() if isinstance(v, float)}
    for name, m in models.items()
}).T
print(comparison.to_string())

In [ ]:
# Bar chart comparison
metric_keys = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
display_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = list(models.keys())
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

x = np.arange(len(metric_keys))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [models[name].get(k, 0) for k in metric_keys]
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(display_names, fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
all_preds = {'Random Forest': rf_pred, 'XGBoost': xgb_pred, 'DNN': dnn_pred, 'Ensemble': ens_pred}

for ax, (name, preds) in zip(axes.flat, all_preds.items()):
    labels = sorted(np.unique(np.concatenate([y_test, preds])))
    names = [ID_TO_CLASS.get(l, str(l)) for l in labels]
    cm = confusion_matrix(y_test, preds, labels=labels)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_title(f'{name}', fontsize=12)
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices (Normalized)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Save Models

In [ ]:
import joblib

model_dir = os.path.join(project_root, 'models')
os.makedirs(model_dir, exist_ok=True)

joblib.dump(rf, os.path.join(model_dir, RF_MODEL_FILE))
joblib.dump(xgb, os.path.join(model_dir, XGB_MODEL_FILE))
dnn.save(os.path.join(model_dir, DNN_MODEL_FILE))
joblib.dump(meta, os.path.join(model_dir, META_MODEL_FILE))

print('All models saved!')